# AI_TalentMatcher: free LLM server on a Kaggle GPU

This notebook runs an open-source LLM with **Ollama** on Kaggle's free GPU. It exposes the model to
your laptop as an **OpenAI-compatible API**, protected by a secret token, through a free
**Cloudflare tunnel**:

```
your app (Docker, LiteLLM) ──HTTPS──► *.trycloudflare.com ──► token proxy :8080 ──► Ollama :11434 (GPU)
```

**Before running:**
1. Kaggle account with a **verified phone number** (required for GPU and Internet).
2. Right panel → **Session options**: *Accelerator* = **GPU T4 x2**, *Internet* = **On**.
3. Run the cells **in order** in an *interactive* session (don't "Save & Run All").

**Limits to know:**
- The GPU has a weekly quota (see your Kaggle settings).
- A session lasts at most ~12 hours and stops if left idle.
- **The tunnel URL and token change every session**, so paste the new `.env` block each time.

**Security:** the token is printed below. Never make this notebook public, and never save a version with outputs.

## 1. Settings

In [ ]:
MODEL = "qwen2.5:7b"     # good at JSON, ~4.7 GB. Alternatives: "qwen2.5:14b" (better, slower), "llama3.1:8b"
CONTEXT_LENGTH = 8192    # Ollama's default is small: long CVs/many profiles would be silently truncated
PARALLEL = 2             # requests served at the same time (match WORKER_CONCURRENCY in .env)

## 2. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Install Ollama and cloudflared (~1–2 min)

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
!ollama --version && cloudflared --version

## 4. Start the Ollama server

In [ ]:
import os, subprocess, time, urllib.request, urllib.error

def wait_http(url, timeout=120):
    """Wait until something answers on `url` (any HTTP status counts as 'up')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            urllib.request.urlopen(url, timeout=5)
            return True
        except urllib.error.HTTPError:
            return True
        except Exception:
            time.sleep(1)
    raise RuntimeError(f"Nothing answered on {url} after {timeout}s")

ollama_env = {
    **os.environ,
    "OLLAMA_HOST": "127.0.0.1:11434",        # local only: the proxy is the sole way in
    "OLLAMA_CONTEXT_LENGTH": str(CONTEXT_LENGTH),
    "OLLAMA_NUM_PARALLEL": str(PARALLEL),
    "OLLAMA_KEEP_ALIVE": "-1",               # keep the model in GPU memory
}
ollama_proc = subprocess.Popen(["ollama", "serve"], env=ollama_env,
                               stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
wait_http("http://127.0.0.1:11434/api/version")
print("Ollama is running")

## 5. Download the model and load it on the GPU (~2–5 min)

In [ ]:
!ollama pull {MODEL}

import json
t0 = time.time()
warmup = urllib.request.Request(
    "http://127.0.0.1:11434/api/generate",
    data=json.dumps({"model": MODEL, "prompt": "Say OK.", "stream": False}).encode(),
    headers={"Content-Type": "application/json"},
)
print(json.load(urllib.request.urlopen(warmup, timeout=600))["response"], f"({time.time() - t0:.0f}s)")
!ollama ps

## 6. Token proxy (Ollama itself has no authentication)

In [ ]:
%%writefile /kaggle/working/auth_proxy.py
"""Tiny bearer-token proxy in front of Ollama (standard library only).

Ollama has no authentication. The Cloudflare tunnel makes it reachable from the internet,
so this proxy rejects every request that doesn't carry `Authorization: Bearer <PROXY_TOKEN>`.
That's the same header LiteLLM sends as `api_key` for OpenAI-compatible endpoints.

    PROXY_TOKEN=... python auth_proxy.py      # listens on 127.0.0.1:8080 -> Ollama 127.0.0.1:11434
"""

import hmac
import http.server
import json
import os
import urllib.error
import urllib.request

TOKEN = os.environ["PROXY_TOKEN"]
UPSTREAM = os.getenv("PROXY_UPSTREAM", "http://127.0.0.1:11434")
HOST = os.getenv("PROXY_HOST", "127.0.0.1")
PORT = int(os.getenv("PROXY_PORT", "8080"))


class Handler(http.server.BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def _send(self, code: int, body: bytes, content_type: str = "application/json") -> None:
        self.send_response(code)
        self.send_header("Content-Type", content_type or "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def _error(self, code: int, message: str) -> None:
        self._send(code, json.dumps({"error": {"message": message}}).encode())

    def _forward(self, method: str) -> None:
        length = int(self.headers.get("Content-Length") or 0)
        body = self.rfile.read(length) if length else None  # always drain the request body

        supplied = self.headers.get("Authorization", "").encode()
        if not hmac.compare_digest(supplied, f"Bearer {TOKEN}".encode()):  # constant-time compare
            return self._error(401, "Invalid or missing API token.")

        request = urllib.request.Request(
            UPSTREAM + self.path,
            data=body,
            method=method,
            headers={"Content-Type": self.headers.get("Content-Type", "application/json")},
        )
        try:
            with urllib.request.urlopen(request, timeout=600) as resp:
                self._send(resp.status, resp.read(), resp.headers.get("Content-Type"))
        except urllib.error.HTTPError as exc:  # Ollama answered with an error: pass it through
            self._send(exc.code, exc.read(), exc.headers.get("Content-Type"))
        except Exception as exc:  # Ollama not running, crashed, ...
            self._error(502, f"Upstream model server unavailable: {exc}")

    def do_GET(self):
        self._forward("GET")

    def do_POST(self):
        self._forward("POST")

    def log_message(self, fmt, *args):  # one short line per request, never the headers
        print(f"[proxy] {self.address_string()} {fmt % args}", flush=True)


if __name__ == "__main__":
    print(f"[proxy] listening on {HOST}:{PORT} -> {UPSTREAM}", flush=True)
    http.server.ThreadingHTTPServer((HOST, PORT), Handler).serve_forever()

In [ ]:
import secrets
PROXY_TOKEN = secrets.token_urlsafe(32)
proxy_proc = subprocess.Popen(["python", "/kaggle/working/auth_proxy.py"],
                              env={**os.environ, "PROXY_TOKEN": PROXY_TOKEN},
                              stdout=open("/kaggle/working/proxy.log", "w"), stderr=subprocess.STDOUT)
wait_http("http://127.0.0.1:8080/")
print("Proxy is running")

## 7. Public HTTPS tunnel (free, no Cloudflare account needed)

In [ ]:
import re
tunnel_log = "/kaggle/working/cloudflared.log"
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8080"],
                               stdout=open(tunnel_log, "w"), stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(60):
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(tunnel_log).read())
    if found:
        PUBLIC_URL = found.group(0)
        break
    time.sleep(1)
assert PUBLIC_URL, "Tunnel did not start; see /kaggle/working/cloudflared.log"
print("Public URL:", PUBLIC_URL)

## 8. Test through the tunnel, exactly as the app will call it

In [ ]:
def chat_via_tunnel(token):
    req = urllib.request.Request(
        f"{PUBLIC_URL}/v1/chat/completions",
        data=json.dumps({
            "model": MODEL,
            "messages": [{"role": "user", "content": 'Return the JSON object {"ok": true} and nothing else.'}],
            "response_format": {"type": "json_object"},
            "temperature": 0,
        }).encode(),
        headers={"Content-Type": "application/json", "Authorization": f"Bearer {token}"},
    )
    return urllib.request.urlopen(req, timeout=120)

for attempt in range(10):  # the new tunnel hostname can take a few seconds to resolve
    try:
        t0 = time.time()
        answer = json.load(chat_via_tunnel(PROXY_TOKEN))["choices"][0]["message"]["content"]
        print(f"OK in {time.time() - t0:.1f}s ->", answer)
        break
    except Exception as exc:
        print("waiting for the tunnel...", type(exc).__name__)
        time.sleep(3)

try:
    chat_via_tunnel("wrong-token")
    print("WARNING: a wrong token was accepted!")
except urllib.error.HTTPError as exc:
    print("Wrong token correctly rejected:", exc.code)

## 9. Paste this into `ai_talentmatcher/.env`, then run `docker compose up --build`

In [ ]:
print(f"""# --- Kaggle GPU (valid for this session only)
LLM_MODEL=openai/{MODEL}
LLM_API_BASE={PUBLIC_URL}/v1
LLM_API_KEY={PROXY_TOKEN}
LLM_REQUESTS_PER_MINUTE=0
WORKER_CONCURRENCY={PARALLEL}
LLM_TIMEOUT_SECONDS=120
""")

## 10. Keep the server alive

Run this cell and **keep the browser tab open**: Kaggle stops idle sessions.
Interrupt it (■) when you're done, then **stop the session** to save GPU quota.

In [ ]:
try:
    while True:
        status = {name: ("up" if p.poll() is None else f"DOWN (exit {p.returncode})")
                  for name, p in [("ollama", ollama_proc), ("proxy", proxy_proc), ("tunnel", tunnel_proc)]}
        requests_seen = sum(1 for line in open("/kaggle/working/proxy.log") if "POST" in line)
        print(time.strftime("%H:%M:%S"), status, f"| requests served: {requests_seen}", flush=True)
        time.sleep(60)
except KeyboardInterrupt:
    print("Monitoring stopped (the server keeps running until the session ends).")